# 06 — N64 reconstruction: 64-bubble GW spectrum comparison

Compare the 3D sledgehamr run (64 bubbles, L=200, λ̄=0.84) with the
spectrum reconstructed from the 1D BM scan × collision weights.

Two comparison times:
- **T₁**: last available SH snapshot
- **T₂**: end of the reconstruction window (the transition is complete)

**Workflow**
1. Run **Section 1** to write the weights input and SLURM script.
2. Submit `sbatch scripts/run_weights_N64.slurm` and wait for it to finish.
3. Continue from **Section 2** to load the result and make the plot.

In [ ]:
import glob
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.physics import InstantonProfile, BubbleKinematics
from ptbuilder.analysis import write_weights_input, load_weights, apply_keffsq
from ptbuilder.scan import load_scan
from ptbuilder.ic import _cutoff_times

# 64 bubble positions from reference_codes/notebooks/Scan.ipynb
bubbles = np.array([[  0.        ,   0.        ,   0.        ],
       [104.21080818, 134.00331875, 115.32260336],
       [ 82.91633086,  40.22040859, 161.10457444],
       [162.20889478, 164.79584079,   2.8384039 ],
       [154.47733984, 187.65793396,  20.16866321],
       [199.13096101,  27.01104231,  45.30013382],
       [168.5003722 ,  38.64441063,  52.81041635],
       [133.97360158,  91.59693268, 138.34643773],
       [ 50.28297449,  51.95537184,  23.62294384],
       [177.13769118, 134.66889799,   8.71668886],
       [ 30.68289591,   9.45788771,  50.50424856],
       [ 64.12769861, 195.12306707,  81.57164046],
       [ 56.68936652, 130.84497227, 108.2354709 ],
       [ 82.99174886, 121.03920058, 172.37415331],
       [127.09687502, 120.51398912, 134.53558472],
       [ 33.36926752,  13.63720659,   0.60996262],
       [ 18.39375397, 191.40492629, 122.47329764],
       [ 19.63641252,  63.09685344, 174.17147189],
       [ 90.74169878, 154.13735512, 104.39306946],
       [ 86.557716  ,  72.91515918, 182.95789915],
       [119.33310673,  37.76924713, 149.28485556],
       [ 17.13480987, 132.62322674, 189.63031283],
       [109.46636148, 118.08537517,  76.31000499],
       [170.09845239,   3.4854559 ,  23.50245978],
       [ 18.70476325, 147.66938079,  58.80734523],
       [111.64277024, 157.053698  , 132.24005744],
       [ 42.14101276,  50.93855746, 176.75407925],
       [ 14.51198083,  16.93867057,  60.2360491 ],
       [ 52.77856703,  29.30614115,  36.71599   ],
       [ 38.07656718,  13.10864189,  72.07363165],
       [138.51107657, 177.90126384, 114.77664924],
       [ 25.84809123, 112.82413992,  33.27661835],
       [  6.6733563 , 160.69190438, 139.70034561],
       [197.70475434,  13.8230847 ,  60.37034021],
       [ 12.25179549, 100.55314862,  78.92943814],
       [140.93499401, 164.76389065, 124.56483354],
       [100.40316917,  68.76075668,  17.56098047],
       [144.15542294,  50.57241845,  83.47993277],
       [ 43.42498771,  75.67311377, 122.44988237],
       [ 73.56937364,  38.65136447,   0.68298585],
       [160.40914341,   1.50148649,   2.225444  ],
       [127.8706212 ,  29.43082072,  23.094503  ],
       [ 10.24110325, 111.83791255, 156.45308871],
       [161.49404006, 163.38664412, 161.98093237],
       [ 27.04229405,  45.36589381, 117.35604037],
       [126.85079112, 162.26356355,  70.60492779],
       [128.66689612,  68.71766065, 150.35375572],
       [166.89624473, 124.99140447,  49.97804878],
       [ 22.58882999, 172.17683864,  41.74842148],
       [173.1491633 , 103.35193697,  79.45686904],
       [ 63.47467018, 157.89290094,  54.88476635],
       [ 83.264726  , 141.90140818, 151.48688241],
       [ 71.99367711,  16.40398225, 197.67281455],
       [ 55.41434491, 148.76063021,   3.30900276],
       [136.79240841,  60.8792625 , 193.43780416],
       [ 21.14423948,  15.16246438, 137.40606156],
       [ 37.08759453,  55.1292117 ,  75.06349727],
       [  6.48071698, 123.73465632,   4.00908165],
       [ 49.63963798,  63.59991319, 136.27840358],
       [  6.47400652, 161.05000329, 163.66077338],
       [142.56695796, 110.77450728, 194.29867779],
       [ 85.21456401,  72.43223496, 118.71280211],
       [103.95645911,   4.65524665, 154.29805374],
       [151.90410329, 147.74780743,  72.22549058]])

In [ ]:
REPO_ROOT   = Path(pt.__file__).parent.parent
SH_RUNS_DIR = REPO_ROOT / 'sledgehamr_runs'
DATA_DIR    = REPO_ROOT / 'data'

lb         = 0.84
L          = 200.
zero_pad   = 1.
N_WEIGHTS  = 1024
T_SCAN_MAX = 70.
N_SCAN     = 32

WEIGHTS_IN  = DATA_DIR / 'weights_in_N64.h5'
WEIGHTS_OUT = DATA_DIR / 'weights_out_N64.h5'
SH_PATH     = SH_RUNS_DIR / 'bubbles_N64'

scan_times    = np.linspace(1., T_SCAN_MAX, N_SCAN)
weights_times = np.linspace(1., T_SCAN_MAX, N_WEIGHTS)

# Mean bubble separation for this simultaneous-nucleation realization --
# hoisted here (rather than only in the CHW section below) so it's
# available for masking the new scan's valid kR_* range as early as
# Section 5.
Rstar = (L**3 / len(bubbles))**(1.0 / 3.0)

# The gs1-20 scan (19_production_scan.ipynb) was only built with guaranteed
# frequency coverage for kR_* in this range (its own KR_MIN/KR_MAX) -- the
# new-scan curve must be masked to it wherever it's plotted.
NEW_SCAN_KR_MIN, NEW_SCAN_KR_MAX = 1.0, 100.0

## 1. Set up weights calculation

Writes the HDF5 input file for the `cpp/weights` binary and generates a shell script.

In [ ]:
def load_kinematics_local(lb):
    """Load BubbleKinematics from stored instanton.h5 (no C++ needed)."""
    path = DATA_DIR / f'phi4_lambda_bar{lb}' / 'instanton.h5'
    with h5py.File(path, 'r') as f:
        profile = InstantonProfile(
            R=f['R'][:], Phi=f['Phi'][:],
            rin_0 =float(f.attrs['rin_0']),
            rmid_0=float(f.attrs['rmid_0']),
            rout_0=float(f.attrs['rout_0']),
            interp=None,
        )
    return BubbleKinematics(profile)


kin = load_kinematics_local(lb)

write_weights_input(
    path=WEIGHTS_IN,
    positions=bubbles,
    t=weights_times,
    kinematics=kin,
    L=L,
    collision_radius='mid',
)
print(f'Written: {WEIGHTS_IN.name}')

weights_dir = REPO_ROOT / 'cpp/weights'
weights_bin = weights_dir / 'weights'
scripts_dir = REPO_ROOT / 'scripts'
logs_dir = REPO_ROOT / 'logs'
scripts_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)

slurm_path = scripts_dir / 'run_weights_N64.slurm'
slurm_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=weights_N64
#SBATCH --output={logs_dir}/weights_N64_%j.out
#SBATCH --constraint=cpu
#SBATCH --qos=regular
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=128
#SBATCH --time=00:10:00

set -euo pipefail
make -C {weights_dir}
export OMP_NUM_THREADS=${{SLURM_CPUS_PER_TASK}}
export OMP_PLACES=threads
export OMP_PROC_BIND=spread
srun {weights_bin} {WEIGHTS_IN} {WEIGHTS_OUT}
''')
print(f'Written: {slurm_path}')
print(f'Submit with:  sbatch {slurm_path}')

## 2. Load scan results

In [ ]:
cache = DATA_DIR / f'phi4_lambda_bar{lb}' / 'scan_cache.h5'
scan_results = load_scan(cache)
actual_t = next(iter(scan_results.values())).times
print(f'lb={lb}: {len(scan_results)} gamma values  '
      f't=[{actual_t[0]:.1f}, {actual_t[-1]:.1f}]  n_t={len(actual_t)}')

### 2b. Load the new gs1-20 production scan

Self-contained, non-rectangular scan (`19_production_scan.ipynb`) -- each
`gamma_ij` row has its own `times`/`wlist`, so the old scan's
`RegularGridInterpolator` approach doesn't apply. `new_interp_fn` mirrors
`16_n64_reconstruction_scan_comparison.ipynb`'s `new_interp_fn` exactly
(log-linear across each row's own `times`, log-linear across the two
bracketing `gamma_ij` rows), projected onto the OLD scan's own `k_out` grid
so both reconstructions land on directly comparable frequency axes.

In [ ]:
from scipy.interpolate import interp1d
from ptbuilder.analysis import load_scan as load_scan_new

# k_out: this scan's own w grid -- also (re)computed in cell-13's
# build_log_scan_interpolator; same value, kept independent here so this
# cell doesn't depend on cell-13's ordering.
k_out = next(iter(scan_results.values())).w
Nk = len(k_out)

NEW_SCAN_DIR = DATA_DIR / 'runtime_scan_lb0.84_gs1-20_kR1-100_nw32'
new_scan = load_scan_new(NEW_SCAN_DIR)
new_gammas = np.array([r.gamma_ij for r in new_scan.rows])
scan_t_new = np.array([min(r.times[0] for r in new_scan.rows), new_scan.t_max])
print(f'new scan: {len(new_scan.rows)} rows, gamma_ij {new_gammas[0]:.2f}..{new_gammas[-1]:.2f}, '
      f'T_MAX={new_scan.t_max:.2f}')

# Project each row onto the shared k_out grid, then build a per-row
# log-time interpolator: floor below t_first, clip to the last value at/
# above T_MAX.
new_spec_on_kout = []
for r in new_scan.rows:
    spec_k = np.zeros((len(r.times), Nk))
    for it in range(len(r.times)):
        spec_k[it] = np.interp(k_out, r.wlist, r.spectrum[it], left=0., right=0.)
    new_spec_on_kout.append(spec_k)

new_floor = max(s.max() for s in new_spec_on_kout) * 1e-12
new_log_floor = np.log(new_floor)

new_row_log_interp = []
for r, spec_k in zip(new_scan.rows, new_spec_on_kout):
    log_spec = np.log(np.maximum(spec_k, new_floor))
    new_row_log_interp.append(interp1d(
        r.times, log_spec, axis=0, kind='linear', bounds_error=False,
        fill_value=(new_log_floor, log_spec[-1]),
    ))


def new_interp_fn(pts):
    g_arr, t_arr = pts[:, 0], pts[:, 1]
    out = np.empty((len(pts), Nk))
    for g_val in np.unique(g_arr):
        sel = g_arr == g_val
        g_clip = float(np.clip(g_val, new_gammas.min(), new_gammas.max()))
        ig_hi = int(np.searchsorted(new_gammas, g_clip))
        ig_hi = min(max(ig_hi, 1), len(new_gammas) - 1)
        ig_lo = ig_hi - 1
        g_lo, g_hi = new_gammas[ig_lo], new_gammas[ig_hi]
        frac = 0.0 if g_hi == g_lo else (g_clip - g_lo) / (g_hi - g_lo)
        log_lo = new_row_log_interp[ig_lo](t_arr[sel])
        log_hi = new_row_log_interp[ig_hi](t_arr[sel])
        out[sel] = np.exp(log_lo + frac * (log_hi - log_lo))
    return out

## 3. Load collision weights

Continue after the Section 1 SLURM job has completed.

In [ ]:
if not WEIGHTS_OUT.exists():
    print('Weights not found — rerun Section 1')
else:
    weights, pair_i, pair_j, gammas = load_weights(WEIGHTS_OUT, N_WEIGHTS)
    n_pairs = len(pair_i)
    print(f'{n_pairs} colliding pairs')

    pair_d  = np.zeros(n_pairs)
    pair_tm = np.zeros(n_pairs)
    for p in range(n_pairs):
        i, j = int(pair_i[p]), int(pair_j[p])
        dp = bubbles[i] - bubbles[j]
        dp -= L * np.round(dp / L)
        d = float(np.linalg.norm(dp))
        pair_d[p]  = d
        pair_tm[p] = float(_cutoff_times(d, 0, 1.)[2])   # BM comparison time ≈ 1.095*d

    print(f'd     range: [{pair_d.min():.2f}, {pair_d.max():.2f}]')
    print(f'gamma range: [{gammas.min():.4f}, {gammas.max():.4f}]')
    print(f't_m   range: [{pair_tm.min():.2f}, {pair_tm.max():.2f}]')

## 4. Load sledgehamr 3D spectrum

In [ ]:
def _list_gw_files(sh_path):
    pattern = os.path.join(sh_path, 'gw_spectra', '*', 'spectra.hdf5')
    entries = []
    for fp in glob.glob(pattern):
        dname = os.path.basename(os.path.dirname(fp))
        if dname.isdigit():
            entries.append((int(dname), fp))
    entries.sort(key=lambda x: x[0])
    return entries


def _read_gw_file(fp):
    with h5py.File(fp, 'r') as f:
        k_int = np.asarray(f['k' if 'k' in f else 'k_sq'][:])
        spec  = np.asarray(f['Spectrum' if 'Spectrum' in f else 'spectrum'][:])
        hdr   = np.asarray(f['Header'][:]).reshape(-1)
        t     = float(hdr[0])
    return apply_keffsq(k_int), spec, t


def load_sh_all_snaps(sh_path):
    files = _list_gw_files(sh_path)
    if not files:
        raise FileNotFoundError(f'No gw_spectra/*/spectra.hdf5 in {sh_path}')
    snaps = [_read_gw_file(fp) for _, fp in files]
    snaps.sort(key=lambda x: x[2])
    return snaps


def normalize_sh(k_sq, spectrum, L, zero_pad):
    if k_sq[0] == 0:
        k_sq     = k_sq[1:]
        spectrum = spectrum[1:]
    Leff = L * zero_pad
    r    = np.sqrt(k_sq)
    k    = r * (2 * np.pi / Leff)
    norm = 4.0 * (2 * np.pi * r) * Leff**3
    return k, spectrum * norm


sh_snaps = load_sh_all_snaps(SH_PATH)
t0 = sh_snaps[0][2]
t1 = sh_snaps[-1][2]
print(f'{len(sh_snaps)} snapshots, t=[{t0:.2f}, {t1:.2f}]')

## 5. Reconstruct and compare

In [ ]:
from scipy.interpolate import RegularGridInterpolator


def build_log_scan_interpolator(scan_results, k_out):
    gammas = np.array(sorted(scan_results.keys()))
    times  = next(iter(scan_results.values())).times
    Ng, Nt, Nk = len(gammas), len(times), len(k_out)

    spec_grid = np.zeros((Ng, Nt, Nk))
    for ig, gamma in enumerate(gammas):
        res = scan_results[gamma]
        for it in range(Nt):
            spec_grid[ig, it] = np.interp(k_out, res.w, res.spectrum[it],
                                           left=0., right=0.)

    floor     = spec_grid.max() * 1e-12
    log_grid  = np.log(np.maximum(spec_grid, floor))
    log_floor = np.log(floor)

    _raw = RegularGridInterpolator(
        (gammas, times), log_grid,
        bounds_error=False, fill_value=log_floor,
    )

    def interp_fn(pts):
        return np.exp(_raw(pts))

    return interp_fn, gammas, times, k_out


def bm_reconstruct(gamma, weights_t, weights_times, t_max, interp_fn, times, gammas):
    """
    Reconstruct dE/dlnk from the 1D BM scan for one pair.

    Steps at the FINE weights_times resolution, not the coarse (32-point)
    scan grid: the collision weight can vary much faster in time (brief
    collision windows) than the spectrum (smooth, slowly-varying in log), so
    the reconstruction must be time-stepped at the weights' resolution, with
    the spectrum freshly interpolated from the scan at each fine step --
    the scan exists specifically to be interpolated, not to define the
    reconstruction's own time grid. Matches
    reference_codes/notebooks/Scan.ipynb's combined_spectrum loop.
    """
    g = float(np.clip(gamma, gammas.min(), gammas.max()))

    t_hi   = min(t_max, times[-1])
    mask   = (weights_times >= times[0]) & (weights_times <= t_hi)
    t_comp = weights_times[mask]
    if len(t_comp) == 0 or t_comp[-1] < t_hi:
        t_comp = np.append(t_comp, t_hi)

    w = np.interp(t_comp, weights_times, weights_t)

    pts   = np.column_stack([np.full(len(t_comp), g), t_comp])
    specs = interp_fn(pts)

    delta = np.diff(specs, axis=0)
    return (w[:len(t_comp) - 1, None] * delta).sum(axis=0)


def interp_sh_at_t(snaps, t_target):
    ts = np.array([s[2] for s in snaps])
    if t_target <= ts[0]:
        return snaps[0][0], snaps[0][1], ts[0]
    if t_target >= ts[-1]:
        return snaps[-1][0], snaps[-1][1], ts[-1]
    i = int(np.searchsorted(ts, t_target)) - 1
    k_sq, s_a, t_a = snaps[i]
    _,    s_b, t_b = snaps[i + 1]
    alpha = (t_target - t_a) / (t_b - t_a)
    floor = max(s_a.max(), s_b.max()) * 1e-12
    log_a = np.log(np.maximum(s_a, floor))
    log_b = np.log(np.maximum(s_b, floor))
    return k_sq, np.exp(log_a + alpha * (log_b - log_a)), t_target

In [ ]:
# Comparison times.
# Weights are 0 before AND after each collision (they peak during the event).
# T2 = last time any pair is still actively colliding (sum_w > threshold).
T1 = sh_snaps[-1][2]  # last snapshot — upper limit

sum_w  = weights.sum(axis=0)            # (n_weights,) — summed over all pairs
active = np.where(sum_w > 0.01)[0]
T2     = min(float(weights_times[-1]), T1)

print(f'T1 = {T1:.2f}  (last snapshot)')
print(f'T2 = {T2:.2f}  (transition complete)')

# Build scan interpolator
first_res = next(iter(scan_results.values()))
k_out = first_res.w
interp_fn, gammas_grid, scan_t, _ = build_log_scan_interpolator(scan_results, k_out)

# BM reconstruction: bm_reconstruct's t_hi = min(t_max, scan_t[-1]) clamps
# T1 and T2 to the same effective bound (scan_t[-1] = T_SCAN_MAX), and
# weights_times never exceeds T_SCAN_MAX either -- so the reconstruction has
# already fully saturated by T2 (weight ~0 for every pair beyond it) and
# recon(T1) carries no new physics over recon(T2). Compute once at T2 and
# verify that assumption instead of computing (and plotting) both.
recon_T2 = np.zeros(len(k_out))
for p in range(n_pairs):
    recon_T2 += bm_reconstruct(float(gammas[p]), weights[p], weights_times, T2,
                                interp_fn, scan_t, gammas_grid)

recon_T1 = np.zeros(len(k_out))
for p in range(n_pairs):
    recon_T1 += bm_reconstruct(float(gammas[p]), weights[p], weights_times, T1,
                                interp_fn, scan_t, gammas_grid)
assert np.allclose(recon_T1, recon_T2, rtol=1e-6, atol=recon_T2.max() * 1e-9), \
    'recon(T1) and recon(T2) diverge -- the "no new physics beyond T2" assumption may not hold here'
print('Verified: recon(T1) == recon(T2) (no new physics between T2 and T1, as expected)')
recon = recon_T2

# Same reconstruction, fed through the new gs1-20 production scan's
# interp_fn (see Section 2b) instead of the old scan's -- same weights,
# same pairs, same T2, projected onto the same k_out.
recon_new = np.zeros(len(k_out))
for p in range(n_pairs):
    recon_new += bm_reconstruct(float(gammas[p]), weights[p], weights_times, T2,
                                new_interp_fn, scan_t_new, new_gammas)

# The gs1-20 scan only guarantees frequency coverage for kR_* in
# [NEW_SCAN_KR_MIN, NEW_SCAN_KR_MAX] -- outside that, new_interp_fn is
# extrapolating past what the scan was built for, so only plot within it.
new_scan_valid = (k_out * Rstar >= NEW_SCAN_KR_MIN) & (k_out * Rstar <= NEW_SCAN_KR_MAX)

# 3D SH spectra at T1 and T2 -- these ARE genuinely different (real snapshots)
sh_at = {}
for T in [T1, T2]:
    k_sq, spec, t_act = interp_sh_at_t(sh_snaps, T)
    k_sh, y_sh = normalize_sh(k_sq, spec, L, zero_pad)
    sh_at[T] = (k_sh, y_sh, t_act)

fig, ax = plt.subplots(figsize=(8, 5))

k1, y1, ta1 = sh_at[T1]
k2, y2, ta2 = sh_at[T2]

# 3D SH: shaded band between T2 (lower) and T1 (upper)
y2_r = np.interp(k1, k2, y2, left=0., right=0.)
ax.fill_between(k1, y2_r, y1, color='C0', alpha=0.15, linewidth=0)
ax.plot(k2, y2, color='C0', lw=2, linestyle=':', label=rf'3+1D  $t={ta2:.0f}$ (transition complete)')
ax.plot(k1, y1, color='C0', lw=2,              label=rf'3+1D  $t={ta1:.0f}$ (last snapshot)')

# 1+1D reconstruction — single curve (see note above: T1 and T2 give the same result)
ax.plot(k_out, recon, color='black', lw=2, linestyle='--', label='1+1D incoherent recon (old scan)')
ax.plot(k_out[new_scan_valid], recon_new[new_scan_valid], color='crimson', lw=1.8, linestyle='--',
        label='1+1D incoherent recon (gs1-20 scan)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylim(1e3,6e7)
ax.set_xlim(0.02,20)
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$\mathrm{d}E_{\mathrm{GW}}/\mathrm{d}\ln k$')
ax.legend(frameon=False)
ax.set_title(rf'N64 reconstruction — $\bar{{\lambda}}={lb}$')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'n64_reconstruction.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Full late-time 3D summary: every spectrum from the last active collision
# through the final t~90 output. The band is temporal variability.
T_COLL_END = float(weights_times[active[-1]])
late_snaps = [(k_sq, spec, t) for k_sq, spec, t in sh_snaps if t >= T_COLL_END]

k_sq_ref, spec_ref, _ = late_snaps[-1]
k_late_ref, _ = normalize_sh(k_sq_ref, spec_ref, L, zero_pad)
late_3d = []
for k_sq_late, spec_late, t_late in late_snaps:
    k_late, y_late = normalize_sh(k_sq_late, spec_late, L, zero_pad)
    late_3d.append(np.interp(k_late_ref, k_late, y_late, left=np.nan, right=np.nan))
late_3d = np.asarray(late_3d)
late_median = np.nanmedian(late_3d, axis=0)
late_min = np.nanmin(late_3d, axis=0)
late_max = np.nanmax(late_3d, axis=0)

print(f'Late-time band: {len(late_snaps)} spectra, '
      f't={late_snaps[0][2]:.2f} .. {late_snaps[-1][2]:.2f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(k_late_ref, late_min, late_max, color='C0', alpha=0.18, linewidth=0,
                label=rf'3+1D temporal range, $t={late_snaps[0][2]:.1f}$--${late_snaps[-1][2]:.1f}$')
ax.plot(k_late_ref, late_median, color='C0', lw=2.2,
        label=rf'3+1D temporal median ({len(late_snaps)} spectra)')
ax.plot(k_out, recon, color='black', lw=2, linestyle='--',
        label='1+1D incoherent recon')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylim(1e3, 6e7)
ax.set_xlim(0.02, 20)
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$\mathrm{d}E_{\mathrm{GW}}/\mathrm{d}\ln k$')
ax.legend(frameon=False)
ax.set_title(rf'N64 reconstruction — $\bar{{\lambda}}={lb}$')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'n64_reconstruction.pdf', bbox_inches='tight')
plt.show()

# Use the same physical lower bound in the literature-comparison cell below.
T2 = T_COLL_END

## 6. Comparison with Cutting, Hindmarsh & Weir (arXiv:1802.05712)

Put the corrected reconstruction and our 3+1D result in the dimensionless convention of Fig. 15,

$$x=kR_*,\qquad y=\frac{1}{(H_*R_*\Omega_{\rm vac})^2}\frac{d\Omega_{\rm gw}}{d\ln k}.$$

The Cutting curve below is digitized raw $N_b=64$ lattice data, not their fitted black curve. The envelope and bulk-flow curves use Eqs. (41)--(46) of arXiv:2005.13537, which quote the exponential-nucleation fits used in Fig. 10. No amplitude or horizontal adjustment is fitted here.

In [ ]:
Leff  = L * zero_pad


def rho_vac_bar(lambda_bar):
    """False-minus-true vacuum energy in BM dimensionless units."""
    U = 3.0 + np.sqrt(9.0 - 8.0 * lambda_bar)
    V_true = 0.5 - U / (4.0 * lambda_bar) + U**2 / (32.0 * lambda_bar)
    return -V_true


def energy_to_chw(k, dE_dlnk, Rstar, lambda_bar, volume):
    """Convert the repo's total dE/dlnk to the CHW Fig. 15 quantity.

    This is the same normalization derived in the old reference normalization
    notebook, written here directly for the already-normalized dE/dlnk arrays.
    There is no empirical multiplier.
    """
    rho_vac = rho_vac_bar(lambda_bar)
    x = np.asarray(k) * Rstar
    y = (3.0 / (8.0 * np.pi * rho_vac**2 * Rstar**2 * volume)) * np.asarray(dE_dlnk)
    return x, y


def broken_power_law(kR, peak, kR_peak, a, b):
    """Eq. (41), normalized so y(kR_peak) == peak."""
    kR = np.asarray(kR)
    return peak * (a + b) * kR_peak**b * kR**a / (b * kR_peak**(a + b) + a * kR**(a + b))

def smooth_broken_power_law(kR, peak, kR_peak, a, b, c):
    # Eq. (47), normalized so y(kR_peak) == peak.
    kR = np.asarray(kR)
    numerator = (a + b)**c * kR_peak**b * kR**a
    denominator = (b * kR_peak**((a + b) / c) + a * kR**((a + b) / c))**c
    return peak * numerator / denominator

# Eq. (44): beta R_* = (8 pi)^(1/3) for exponential nucleation.
beta_Rstar = (8.0 * np.pi)**(1.0 / 3.0)
kR_models = np.logspace(-2, 4, 1200)

# Eqs. (41)--(43): envelope approximation.
env = broken_power_law(
    kR_models,
    peak=4.7e-2 / beta_Rstar**2,
    kR_peak=1.07 * beta_Rstar,
    a=2.9, b=0.9,
)

# Eqs. (41), (44)--(46): bulk-flow model.
bulk = broken_power_law(
    kR_models,
    peak=6.4e-2 / beta_Rstar**2,
    kR_peak=0.809 * beta_Rstar,
    a=0.9, b=2.1,
)

# Eqs. (47)--(50): full-lattice fit to the 2018 thin-wall simulations.
lattice_fit = smooth_broken_power_law(
    kR_models, peak=3.22e-3, kR_peak=3.20, a=3.0, b=1.51, c=2.18,
)

# Raw points digitized from Fig. 15 in the original reference workflow.
cutting_path = (REPO_ROOT / 'reference_codes' / 'FirstOrder_GW' /
                'collision_weights' / 'Weir1802' / 'fig15_gamma4_b64.txt')
cutting = {nb: np.loadtxt(cutting_path.with_name(f'fig15_gamma4_b{nb}.txt'), delimiter=',')
           for nb in (8, 64, 512, 4096)}
lattice_fit_published = np.loadtxt(cutting_path.with_name('fig15_lattice_fit.txt'), delimiter=',')

# Convert only units. The same fixed conversion is applied to our SH and BM curves.
x_sh1, chw_sh1 = energy_to_chw(k1, y1, Rstar, lb, Leff**3)
x_sh2, chw_sh2 = energy_to_chw(k2, y2, Rstar, lb, Leff**3)
x_recon, chw_recon = energy_to_chw(k_out, recon, Rstar, lb, Leff**3)
x_recon_new, chw_recon_new = energy_to_chw(k_out, recon_new, Rstar, lb, Leff**3)

# Late-time 3D summary: pointwise statistics over all snapshots at t >= T2.
# This describes temporal ringing/variability, not statistical uncertainty.
late_times = np.array([t for _, _, t in sh_snaps if t >= T2])
late_chw = []
for k_sq_late, spec_late, t_late in sh_snaps:
    if t_late < T2:
        continue
    k_late, dE_late = normalize_sh(k_sq_late, spec_late, L, zero_pad)
    x_late, y_late = energy_to_chw(k_late, dE_late, Rstar, lb, Leff**3)
    late_chw.append(np.interp(x_sh1, x_late, y_late, left=np.nan, right=np.nan))
late_chw = np.asarray(late_chw)
chw_late_median = np.nanmedian(late_chw, axis=0)
chw_late_min = np.nanmin(late_chw, axis=0)
chw_late_max = np.nanmax(late_chw, axis=0)

print(f'R_* = {Rstar:.4f},  rho_vac = {rho_vac_bar(lb):.6g}')
print('Literature normalization contains no fitted scale factors.')

fig, ax = plt.subplots(figsize=(8.5, 5.5))

# Our corrected calculation: median and full temporal range for t >= 70.
ax.fill_between(x_sh1, chw_late_min, chw_late_max, color='C0', alpha=0.18, linewidth=0,
                label=rf'our 3+1D temporal range, $t\geq{T2:.0f}$')
ax.plot(x_sh1, chw_late_median, color='C0', lw=2.3,
        label=rf'our 3+1D temporal median, $t\geq{T2:.0f}$')
ax.plot(x_recon, chw_recon, color='black', lw=2.2, ls='--',
        label=rf'our 1+1D reconstruction (old scan), $\bar{{\lambda}}={lb}$')
ax.plot(x_recon_new[new_scan_valid], chw_recon_new[new_scan_valid], color='crimson', lw=2.0, ls='--',
        label=rf'our 1+1D reconstruction (gs1-20 scan), $\bar{{\lambda}}={lb}$')

# Literature: raw lattice points and the two macroscopic models.
for nb, color in zip((8, 64, 512, 4096), ('#f4a3a3', '#e76f6f', '#d62728', '#8b0000')): ax.plot(cutting[nb][:, 0], cutting[nb][:, 1], color=color, lw=1.7, label=rf'Cutting et al. Fig. 15, gamma~4, N_b={nb}')
ax.plot(lattice_fit_published[:, 0], lattice_fit_published[:, 1], color='0.15', lw=1.9, ls='-',
        label='Cutting--Hindmarsh--Weir lattice fit')
ax.plot(kR_models, env, color='0.35', lw=1.8, ls='--',
        label='envelope (exponential nucleation)')
ax.plot(kR_models, bulk, color='0.35', lw=1.8, ls='-.',
        label='bulk flow (exponential nucleation)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(1e-1, 2e3)
ax.set_ylim(1e-6, 1e-2)
ax.set_xlabel(r'$kR_*$')
ax.set_ylabel(r'$[H_*R_*\Omega_{\rm vac}]^{-2},d\Omega_{\rm gw}/d\ln k$')
ax.set_title(rf'N64 literature comparison -- $\bar{{\lambda}}={lb}$')
ax.legend(frameon=False, fontsize=8.5, ncol=2)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'n64_literature_comparison.pdf', bbox_inches='tight')
plt.show()

## Direct comparison with Figure 8

Figure 8 is the simultaneous-nucleation $N_b=64$, $\gamma_*=3.94$ run. Its five lattice times and its realization-matched envelope calculation are extracted directly from the vector PDF. Our realization is overlaid in the same published units, but is not parameter-identical.

In [ ]:
fig8_dir = cutting_path.parent
fig8_lattice = {tr: np.loadtxt(fig8_dir / f'fig8_lattice_tR{tr}.txt', delimiter=',')
                for tr in (1, 2, 4, 6, 8)}
fig8_envelope = np.loadtxt(fig8_dir / 'fig8_envelope_points.txt', delimiter=',')

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.fill_between(x_sh1, chw_late_min, chw_late_max, color='C0', alpha=0.16, linewidth=0,
                label=rf'our 3+1D temporal range, $t\geq{T2:.0f}$')
ax.plot(x_sh1, chw_late_median, color='C0', lw=2.5,
        label='our 3+1D temporal median')
ax.plot(x_recon, chw_recon, color='C0', lw=2.2, ls='--',
        label='our 1+1D reconstruction (old scan)')
ax.plot(x_recon_new[new_scan_valid], chw_recon_new[new_scan_valid], color='crimson', lw=2.0, ls='--',
        label='our 1+1D reconstruction (gs1-20 scan)')

fig8_colors = plt.cm.Reds(np.linspace(0.35, 0.9, len(fig8_lattice)))
for (tr, raw), color in zip(fig8_lattice.items(), fig8_colors):
    ax.plot(raw[:, 0], raw[:, 1], color=color, lw=1.45,
            label=rf'CHW Fig. 8 lattice, $t/R_*={tr}$')
yerr = np.vstack((fig8_envelope[:, 1] - fig8_envelope[:, 2],
                  fig8_envelope[:, 3] - fig8_envelope[:, 1]))
ax.errorbar(fig8_envelope[:, 0], fig8_envelope[:, 1], yerr=yerr,
            fmt='o', ms=3.5, color='black', elinewidth=0.9, capsize=1.5,
            label='CHW Fig. 8 envelope, matched realization')
ax.axvline(2*np.pi, color='0.45', lw=1.0, ls=':', label=r'$kR_*=2\pi$')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(1e-1, 2e3)
ax.set_ylim(1e-6, 1e-2)
ax.set_xlabel(r'$kR_*$')
ax.set_ylabel(r'$[H_*R_*\Omega_{\rm vac}]^{-2}\,d\Omega_{\rm gw}/d\ln k$')
ax.set_title(r'Our $N_b=64$ vs CHW Fig. 8 ($N_b=64$, $\gamma_*=3.94$)')
ax.legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'n64_fig8_comparison.pdf', bbox_inches='tight')
plt.show()